In [1]:
import argparse
import gc
import os
import sys

import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.model_selection import KFold
from tqdm import tqdm

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
%load_ext autoreload
%autoreload 2
    
from drGAT import drGAT
from drGAT.load_data import load_data
from drGAT.metrics import compute_metrics_stats
from drGAT.myutils import get_all_edges_and_labels
from drGAT.sampler import BalancedSampler

/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch_geometric/typing.py:54: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /lib64/libm.so.6: version `GLIBC_2.29' not found (required by /gpfs/gsfs12/users/inouey2/conda/envs/genex/lib/python3.10/site-packages/libpyg.so)
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch_geometric/typing.py:72: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /gpfs/gsfs12/users/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch_scatter/_scatter_cuda.so: undefined symbol: _ZN2at23SavedTensorDefaultHooks11set_tracingEb
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch_geometric/typing.py:99: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. S

In [3]:
method = 'GAT'
data = 'nci'

In [4]:
def suggest_hyperparams(trial, S_d, S_c, S_g):
    hidden1 = trial.suggest_int("hidden1", 256, 1024)
    hidden2 = trial.suggest_int("hidden2", 64, min(512, hidden1))
    hidden3 = trial.suggest_int("hidden3", 32, min(256, hidden2))

    params = {
        "n_drug": S_d.shape[0],
        "n_cell": S_c.shape[0],
        "n_gene": S_g.shape[0],
        "dropout1": trial.suggest_float("dropout1", 0.1, 0.5, step=0.05),
        "dropout2": trial.suggest_float("dropout2", 0.1, 0.5, step=0.05),
        "hidden1": hidden1,
        "hidden2": hidden2,
        "hidden3": hidden3,
        "epochs": 3,
        # trial.suggest_int("epochs", 100, 10000, step=100),
        "heads": trial.suggest_int("heads", 2, 8),
        "activation": trial.suggest_categorical("activation", ["relu", "gelu"]),
        "optimizer": trial.suggest_categorical("optimizer", ["Adam", "AdamW"]),
        "lr": trial.suggest_float("lr", 1e-5, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "scheduler": trial.suggest_categorical("scheduler", [None, "Cosine"]),
        "norm_type": trial.suggest_categorical(
            "norm_type", ["GraphNorm", "BatchNorm", "LayerNorm"]
        ),
        "gnn_layer": method,
    }

    if params["scheduler"] == "Cosine":
        min_epoch_div = max(1, params["epochs"] // 5)
        max_epoch_div = max(min_epoch_div + 1, params["epochs"] // 2)
        params["T_max"] = trial.suggest_int(
            "T_max", low=min_epoch_div, high=max_epoch_div
        )

        if params["T_max"] <= 0:
            raise optuna.TrialPruned(f"Invalid T_max: {params['T_max']}")

    return params

In [5]:
def handle_optuna_errors(e, trial):
    msg = str(e)
    if "CUDA out of memory" in msg:
        print(f"Pruned trial {trial.number}: CUDA OOM")
        with torch.cuda.device("cuda"):
            torch.cuda.empty_cache()
        gc.collect()
        raise optuna.TrialPruned(f"OOM at trial {trial.number}")
    elif "Input contains NaN" in msg:
        print(f"Pruned trial {trial.number}: Input contains NaN")
        raise optuna.TrialPruned(f"NaN input at trial {trial.number}")
    elif isinstance(e, ZeroDivisionError):
        print(f"Pruned trial {trial.number}: ZeroDivisionError in CosineAnnealingLR")
        raise optuna.TrialPruned("ZeroDivisionError in CosineAnnealingLR")
    else:
        print(f"Unexpected error in trial {trial.number}: {msg}")
        raise e

In [9]:
def objective(trial):
    try:
        is_zero_pad = trial.suggest_categorical("is_zero_pad", [True, False])
        (
            drugAct,
            null_mask,
            S_d,
            S_c,
            S_g,
            drug_feature,
            gene_norm_gene,
            gene_norm_cell,
            A_cg,
            A_dg,
        ) = load_data(data, is_zero_pad=is_zero_pad)

        params = suggest_hyperparams(trial, S_d, S_c, S_g)

        all_edges, all_labels = get_all_edges_and_labels(drugAct, null_mask)
        kf = KFold(n_splits=5, shuffle=True, random_state=42)

        true_datas = pd.DataFrame()
        predict_datas = pd.DataFrame()

        for train_idx, test_idx in tqdm(kf.split(all_edges)):
            sampler = BalancedSampler(
                drugAct,
                all_edges,
                all_labels,
                train_idx,
                test_idx,
                null_mask,
                S_d,
                S_c,
                S_g,
                A_cg,
                A_dg,
            )

            (
                _,
                _,
                _,
                true_data,
                predict_data,
                _,
                _,
                _,
                _,
            ) = drGAT.train(sampler, params=params, device=device, verbose=False)

            true_datas = pd.concat(
                [true_datas, pd.DataFrame(true_data).T], ignore_index=True
            )
            predict_datas = pd.concat(
                [predict_datas, pd.DataFrame(predict_data).T], ignore_index=True
            )


        predict_datas.to_csv('tmp.csv')
        

        

        metrics_result = compute_metrics_stats(
            trial=trial,
            true=true_datas,
            pred=predict_datas,
            target_metrics=["AUROC", "AUPR", "F1", "ACC"],
        )
        return tuple(metrics_result["target_values"])

    except Exception as e:
        handle_optuna_errors(e, trial)


In [10]:
study = optuna.create_study(
    directions=["maximize"] * 4,
    sampler=optuna.samplers.NSGAIISampler(),
    pruner=optuna.pruners.HyperbandPruner(),
    storage=f"sqlite:///{method}_{data}_small.sqlite3",
    study_name=method,
    load_if_exists=True,
)
study.optimize(objective, n_trials=1000)

[I 2025-05-05 19:30:05,461] Using an existing study with name 'GAT' instead of creating a new one.


load nci
Done!


0it [00:00, ?it/s]

Using device: cuda


/spin1/home/linux/inouey2/drGAT/drGAT/drGAT.py:278: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if use_autocast else None
1it [00:00,  1.37it/s]

...

/spin1/home/linux/inouey2/drGAT/drGAT/drGAT.py:278: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if use_autocast else None
4it [00:02,  1.38it/s]

Best model found at epoch 3
Using device: cuda


/spin1/home/linux/inouey2/drGAT/drGAT/drGAT.py:278: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if use_autocast else None
5it [00:03,  1.35it/s]
[W 2025-05-05 19:30:45,546] Trial 25 failed with parameters: {'is_zero_pad': True, 'hidden1': 639, 'hidden2': 135, 'hidden3': 125, 'dropout1': 0.2, 'dropout2': 0.1, 'heads': 7, 'activation': 'relu', 'optimizer': 'AdamW', 'lr': 0.006804808649822879, 'weight_decay': 0.00018330736852078394, 'scheduler': None, 'norm_type': 'BatchNorm'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/optuna/study/_optimize.py", line 196, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_258695/3286120031.py", line 65, in objective
    metrics_result = compute_metrics_stats(
  File "/spin1/home/linux/inouey2/drGAT/drGAT/metrics.py", l

Best model found at epoch 3
0       0
1       0
2       0
3       0
4       0
       ..
2858    0
2859    0
2860    0
2861    0
2862    0
Name: 0, Length: 2863, dtype: int64
0       1.0
1       1.0
2       1.0
3       1.0
4       1.0
       ... 
2858    0.0
2859    0.0
2860    0.0
2861    0.0
2862    0.0
Name: 0, Length: 2863, dtype: float64



KeyboardInterrupt



In [24]:
print(set(pd.read_csv('tmp.csv', index_col=0).loc[0]))
print(set(pd.read_csv('tmp.csv', index_col=0).loc[1]))
print(set(pd.read_csv('tmp.csv', index_col=0).loc[2]))
print(set(pd.read_csv('tmp.csv', index_col=0).loc[3]))
print(set(pd.read_csv('tmp.csv', index_col=0).loc[4]))

{0.0, 1.0}
{0.0, 1.0}
{0.0, 1.0, nan}
{0.0, 1.0, nan}
{0.0, 1.0, nan}


In [25]:
pd.read_csv('tmp.csv', index_col=0)

,0,1,2,3,4,5,6,7,8,9,...,2853,2854,2855,2856,2857,2858,2859,2860,2861,2862
0,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0.0
1,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0.0
2,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,NaN
3,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,NaN
4,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,NaN
